In [1]:
import pandas as pd
import geopandas as gpd
from typing import Final
from blocksnet.enums import LandUse


In [2]:
from blocksnet.enums import LandUse

benchmarks_demo = {
    LandUse.RESIDENTIAL: {
        "cost_build": 45_000,
        "price_sale": 140_000,
        "construction_years": 3,
        "sale_years": 4,
        "opex_rate": 800,
        "cost_demolition": 900,  # ₽/м² ориентировочно по России
    },
    LandUse.BUSINESS: {
        "cost_build": 55_000,
        "rent_annual": 25_000,
        "rent_years": 15,
        "construction_years": 3,
        "opex_rate": 1_300,
        "cost_demolition": 900,
    },
    LandUse.RECREATION: {
        "cost_build": 20_000,
        "rent_annual": 7_500,
        "rent_years": 15,
        "construction_years": 3,
        "opex_rate": 1_000,
        "cost_demolition": 900,
    },
    LandUse.SPECIAL: {
        "cost_build": 35_000,
        "rent_annual": 11_000,
        "rent_years": 15,
        "construction_years": 3,
        "opex_rate": 1_500,
        "cost_demolition": 900,
    },
    LandUse.INDUSTRIAL: {
        "cost_build": 38_000,
        "rent_annual": 14_800,
        "rent_years": 15,
        "construction_years": 3,
        "opex_rate": 700,
        "cost_demolition": 900,
    },
    LandUse.AGRICULTURE: {
        "cost_build": 25_000,
        "rent_annual": 6_500,
        "rent_years": 15,
        "construction_years": 3,
        "opex_rate": 300,
        "cost_demolition": 900,
    },
    LandUse.TRANSPORT: {
        "cost_build": 18_000,
        "rent_annual": 8_200,
        "rent_years": 15,
        "construction_years": 3,
        "opex_rate": 600,
        "cost_demolition": 900,
    },
}



In [3]:
basline_blocks = gpd.read_file('../data/gatchina/blocks_clean_gatchina.geojson')

target_id = 86
target_block = basline_blocks.loc[basline_blocks["id"] == target_id]
basline_blocks["is_project"] = False
basline_blocks.loc[basline_blocks["id"] == target_id, "is_project"] = True


In [4]:
basline_blocks.loc[basline_blocks["id"] == target_id].iloc[0]

residential                                                      0.268468
business                                                         0.648297
recreation                                                       0.065871
industrial                                                            0.0
transport                                                         0.01739
special                                                               0.0
agriculture                                                           0.0
land_use                                                 LandUse.BUSINESS
share                                                            0.648297
footprint_area                                               47096.975151
build_floor_area                                            107107.690178
living_area                                                  44241.785252
non_living_area                                              62865.904925
population                            

In [5]:
# from urbanomy.methods.land_value_modeling.ga_mc_optimizer import DistrictProblem
# from catboost import CatBoostRegressor

# model = CatBoostRegressor()
# model.load_model('../data/catboost_land_value_no_services.cbm')  # модель на лог-цене
# print(len(model.feature_names_))

# params_repaired = {
#     'footprint_area': 11837.651977125355,
#     'l': 5.2,
#     'mxi': 0.12,

#     'residential': 0.00,
#     'business': 0.12,
#     'recreation': 0.03,
#     'industrial': 0.65,
#     'transport': 0.12,
#     'special': 0.08,
#     'agriculture': 0.00
# }

# feature_cols = [
# 'residential','business','recreation','industrial','transport','special',
# 'agriculture','land_use','share','footprint_area','build_floor_area',
# 'living_area','non_living_area','population','site_area','fsi','gsi',
# 'mxi','l','morphotype','area_accessibility'
# ]
# cat_features = ['land_use', 'morphotype']
# numeric_feats = [c for c in feature_cols if c not in cat_features]

# site_area = float(basline_blocks.loc[basline_blocks["id"] == target_id, "site_area"].iloc[0])

# problem = DistrictProblem(
#     blocks=basline_blocks,
#     target_id=target_id,
#     model=model,
#     benchmarks=benchmarks_demo,
    
# constraints = {
#     "footprint_area": {"type": "float", "min": 0.0, "max": 0.1 * site_area},
#     "l": {"type": "float", "min": 1.0, "max": 10.0},
#     "mxi": {"type": "float", "min": 0.0, "max": 1.0},

#     "residential": {"type": "float", "min": 0.0, "max": 1.0},
#     "business": {"type": "float", "min": 0.0, "max": 1.0},
#     "recreation": {"type": "float", "min": 0.0, "max": 1.0},
#     "industrial": {"type": "float", "min": 0.0, "max": 1.0},
#     "transport": {"type": "float", "min": 0.0, "max": 1.0},
#     "special": {"type": "float", "min": 0.0, "max": 1.0},
#     "agriculture": {"type": "float", "min": 0.0, "max": 1.0},
# },
    
#     estimator_kwargs={
#         "orig_features": numeric_feats+cat_features,
#         "categorical_features": cat_features,
#     }
# )

# params_repaired = problem._repair_genome(params_repaired)
# params_repaired

In [9]:
params_repaired = {
    'footprint_area': 11837.651977125355,
    'l': 1.8,
    'mxi': 0.05,

    'residential': 0.00,
    'business': 0.05,
    'recreation': 0.02,
    'industrial': 0.78,
    'transport': 0.10,
    'special': 0.05,
    'agriculture': 0.00,

    'share': 0.78,
    'land_use': LandUse.INDUSTRIAL,

    'build_floor_area': 21307.77355882564,
    'living_area': 0.0,
    'non_living_area': 21307.77355882564,

    'population': 0.0,

    'fsi': 0.081,
    'gsi': 0.045138637755124125,

    'morphotype': 'industrial'
}

In [10]:
from urbanomy.methods.land_value_modeling import (
    ScenarioTEPModifier,
)
import pandas as pd


modifier = ScenarioTEPModifier(basline_blocks)
blocks_gen = modifier.apply(target_id, params_repaired)

gen_blocks = pd.DataFrame(blocks_gen)
gen_blocks.loc[gen_blocks["id"] == target_id].iloc[0]

residential                                                              0.0
business                                                                0.05
recreation                                                              0.02
industrial                                                              0.78
transport                                                                0.1
special                                                                 0.05
agriculture                                                              0.0
land_use                                                  LandUse.INDUSTRIAL
share                                                                   0.78
footprint_area                                                  11837.651977
build_floor_area                                                21307.773559
living_area                                                              0.0
non_living_area                                                 21307.773559

In [ ]:
from urbanomy.methods.investment_potential import prepare_investment_input

investment_input = prepare_investment_input(
    gdf = gen_blocks
)

investment_input.head()


2026-03-16 15:20:45.180 | WARNING  | urbanomy.utils.investment_input:prepare_investment_input:274 - prepare_investment_input: колонка 'land_value_before' не найдена; значение оставлено пустым, текущая цена записана в 'land_value_after'.


,land_use,land_value,residential,business,recreation,industrial,transport,special,agriculture,site_area,living_area,non_living_area,build_floor_area,land_use_before,build_floor_area_before
0,LandUse.INDUSTRIAL,7.691950e+08,0.0,31470.117573,7867.529393,170463.136856,31470.117573,20980.078382,0.0,262250.979778,7386.694834,54169.095447,61555.790281,LandUse.BUSINESS,107107.690178


## Инвевстиционная привлекательность

In [ ]:
from urbanomy.methods.investment_potential import InvestmentAttractivenessAnalyzer

an = InvestmentAttractivenessAnalyzer(benchmarks=benchmarks_demo)
summary = an.calculate_investment_metrics(investment_input, discount_rate=0.18)
summary


Project totals:
 • Land area:          262,250.98
 • Built area:         61,555.79
 • Land value:         769,194,999.43
 • Demolition cost:    96,396,921.16
 • Construction cost:  2,268,946,429.76
 • Investment need:    3,134,538,350.35
 • Project NPV:        413,726,663.43
 • Project IRR:        0.21
 • Project PI:         1.15
 • Project PP (yrs):   12.60


,land_use,land_area,built_area,land_value,demolition_cost,construction_cost,investment_need,NPV,IRR,PI,PP_years,EI
0,LandUse.INDUSTRIAL,262250.98,61555.79,7.691950e+08,96396921.16,2.268946e+09,3.134538e+09,4.137267e+08,0.21,1.15,12.6,59.46


In [ ]:
# scn.to_file('../data/blocks_investment.geojson', driver='GeoJSON')